In [0]:
%pip install langchain==0.1.17 langchain-openai==0.1.6
dbutils.library.restartPython()
from langchain.agents import AgentExecutor

In [0]:
from pyspark.sql import functions as F
import json
from pyspark.sql.types import DecimalType
from langchain.tools import tool
from langchain.chat_models import ChatDatabricks
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate

In [0]:
def load_rules(table_name: str):

    print(table_name)
    df = spark.read.table("testunitycatalog.default.data_quality_rules") \
                   .filter(F.col("table_name") == table_name)
    display(df)
    reference_schema = {}
    precision_rules  = {}
    type_rules       = {}
    critical_cols    = []

    for r in df.collect():
        if r.rule_type == "schema":
            reference_schema[r.column_name] = r.rule_value

        elif r.rule_type == "precision":
            val = json.loads(r.rule_value)
            precision_rules[r.column_name] = (val["precision"], val["scale"])

        elif r.rule_type == "datatype":
            type_rules[r.column_name] = r.rule_value
        elif r.rule_type == "critical":
            critical_cols.append(r.column_name)

    print(reference_schema)
    print(precision_rules)
    print(type_rules)
    print(critical_cols)
    
    return reference_schema, precision_rules, type_rules, critical_cols

def run_full_pipeline(
    source_path,
    source_format,
    bronze_table,
    silver_table,
    table_name
):
    (reference_schema,precision_rules,type_rules,#  semantic_rules,
     critical_cols) = load_rules(table_name)

    # df = (spark.read.format(source_format)
    #                 .option("header", "true")
    #                 .option("inferSchema", "true")
    #                 .load(source_path))
    
    # display(df)

    # df.createOrReplaceTempView("staging_table")

    # #Agent 1 validation
    # payload = {
    #     "table_name": "staging_table",
    #     "reference_schema": reference_schema,
    #     "precision_rules": precision_rules,
    #     "type_rules": type_rules,
    #     # "semantic_rules": semantic_rules
    # }

    # print(payload)



run_full_pipeline(
    source_path   = "/Volumes/testunitycatalog/bronze/datavolume/orders_samples.csv",
    source_format = "csv",
    bronze_table  = "testunitycatalog.bronze.orders",
    silver_table  = "testunitycatalog.silver.orders",
    table_name    = "orders"
)